Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

    Tracking agent behavior with logging, analytics, and debugging.
    Transforming prompts, tool selection, and output formatting.
    Adding retries, fallbacks, and early termination logic.
    Applying rate limits, guardrails, and PII detection.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-120b")

Summarization middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

    Long-running conversations that exceed context windows.
    Multi-turn dialogues with extensive history.
    Applications where preserving full conversation context matters.


In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### message based summarization middleware
agent = create_agent(
    model=model,    
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model=model,
        trigger=("messages", 10),
        keep=("messages", 4)
    ),
    ],
)

In [3]:
config={"configurable":{"thread_id":"test-1"}}
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='a565326e-0277-49b6-b9e0-ae153e9ea86a'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question. The system says "You are ChatGPT...". There\'s no policy violation. Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 78, 'total_tokens': 126, 'completion_time': 0.099625626, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.003320823, 'prompt_tokens_details': None, 'queue_time': 0.213021985, 'total_time': 0.102946449}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86dc843b12', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1444d-179a-7bd0-ba97-174cfbfc3bcc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 48, 'total_token

In [11]:
### Based on TokenSize based summarization middleware
from langchain.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    tools=[search_hotels],
    middleware=[SummarizationMiddleware(
        model=model,
        trigger=("tokens", 500),
        keep=("tokens", 200)
    ),
    ],
)

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [12]:
cities = ["Paris", "London"]

for city in cities:
    config = {"configurable": {"thread_id": f"test-{city}-1"}}
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~489 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='16a27400-5949-4101-9148-30d57f6e6544'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants hotels in Paris. We have a function to search hotels: search_hotels(city). Use it.', 'tool_calls': [{'id': 'fc_93375b87-fe5a-45fc-a246-b0c5e149fc92', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 129, 'total_tokens': 181, 'completion_time': 0.109162698, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.02170126, 'prompt_tokens_details': None, 'queue_time': 0.310762173, 'total_time': 0.130863958}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_96d96a151c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1445f-ab90-71

Fraction

In [15]:
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)
for city in cities:
    config = {"configurable": {"thread_id": f"test-{city}-1"}}
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~374 tokens (0.2922%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='978cbcbe-a43e-4772-9ec5-8f6623583242'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants "Hotels in Paris". Likely they want a list of hotels. We can call search_hotels function with city "Paris".', 'tool_calls': [{'id': 'fc_12028b4f-d22a-47c7-ab8d-c7d32a31cbb6', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 128, 'total_tokens': 186, 'completion_time': 0.122405295, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.008047696, 'prompt_tokens_details': None, 'queue_time': 0.213122361, 'total_time': 0.130452991}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86dc843b12', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, i


Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

    - High-stakes operations requiring human approval (e.g. database writes, financial transactions).
    - Compliance workflows where human oversight is mandatory.
    - Long-running conversations where human feedback guides the agent.



In [16]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

In [17]:
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [18]:
agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [34]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [20]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='041e59db-87bc-402b-a3c7-5d715d10392b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide parameters.', 'tool_calls': [{'id': 'fc_1ac55ab0-749a-45d7-ab8d-8ecec3134ae1', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.127335201, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.006929775, 'prompt_tokens_details': None, 'queue_time': 0.39072348, 'total_time': 0.134264976}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': Non

In [21]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been sent to **john@test.com** with the subject **“Hello”** and the body **“How are you?”**. Let me know if you need anything else!


In [22]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='041e59db-87bc-402b-a3c7-5d715d10392b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide parameters.', 'tool_calls': [{'id': 'fc_1ac55ab0-749a-45d7-ab8d-8ecec3134ae1', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.127335201, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.006929775, 'prompt_tokens_details': None, 'queue_time': 0.39072348, 'total_time': 0.134264976}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': Non

In [28]:
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I’m ready to send that email, but it looks like the previous attempt was cancelled. Would you like me to go ahead and send the message to **john@test.com** with the subject **“Hello”** and body **“How are you?”**? Just let me know!


In [29]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='041e59db-87bc-402b-a3c7-5d715d10392b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide parameters.', 'tool_calls': [{'id': 'fc_1ac55ab0-749a-45d7-ab8d-8ecec3134ae1', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.127335201, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.006929775, 'prompt_tokens_details': None, 'queue_time': 0.39072348, 'total_time': 0.134264976}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': Non

In [35]:
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: I’m unable to send the email exactly as you wrote it because the system’s reviewer has overridden the request and redirected the message to a different address. The email that was actually sent went to **correct@email.com** with the subject **“Corrected Subject”** and the body **“This was edited by human before sending.”**

If you’d like me to try sending a new message to **john@test.com** (or to any other address) with different content, just let me know the exact details you want, and I’ll proceed.


In [36]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='041e59db-87bc-402b-a3c7-5d715d10392b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide parameters.', 'tool_calls': [{'id': 'fc_1ac55ab0-749a-45d7-ab8d-8ecec3134ae1', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.127335201, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.006929775, 'prompt_tokens_details': None, 'queue_time': 0.39072348, 'total_time': 0.134264976}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': Non